# 🏎️ F1 2025 Championship Prediction Model

This notebook creates a comprehensive machine learning model to predict the Formula 1 2025 World Championship winner using historical race data, driver performance metrics, and advanced feature engineering techniques.

## Objective
Develop an ensemble machine learning model that can accurately predict championship probabilities for F1 drivers in the 2025 season based on historical performance data from 2010-2024.

## Methodology
1. **Data Collection**: Historical F1 data from Ergast API (2010-2024)
2. **Feature Engineering**: Create 30+ meaningful features from raw race data
3. **Model Training**: Ensemble of 7 different ML algorithms
4. **Evaluation**: Cross-validation and performance metrics
5. **Prediction**: 2025 championship probabilities for all drivers

## 1. Import Required Libraries

Import all necessary libraries for data manipulation, machine learning, and visualization.

In [1]:
# Data manipulation and analysis
import pandas as pd
import numpy as np
import sys
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
import joblib

# Advanced ML libraries
import xgboost as xgb
import lightgbm as lgb

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical libraries
from scipy import stats

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

# Add src directory to path for custom modules
sys.path.append('../src')

print("✅ All libraries imported successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🤖 Scikit-learn available")
print(f"📈 Visualization libraries ready")

ModuleNotFoundError: No module named 'plotly'

## 2. Data Collection and Loading

Load historical F1 data using our custom data collector module. This will fetch data from the Ergast API covering seasons 2010-2024.

In [ ]:
# Import our custom modules
try:
    from data_collector import F1DataCollector
    from feature_engineering import F1FeatureEngineer
    from ml_model import F1ChampionshipPredictor
    from model_evaluator import F1ModelEvaluator
    print("✅ Custom modules imported successfully!")
except ImportError as e:
    print(f"❌ Error importing custom modules: {e}")
    print("Make sure you're running this notebook from the notebooks directory")

# Initialize data collector
collector = F1DataCollector()
print("🏎️ F1 Data Collector initialized")

# Try to load existing data first, otherwise collect new data
data_file = '../data/f1_championship_data.csv'

if os.path.exists(data_file):
    print("📂 Loading existing F1 data...")
    df_raw = pd.read_csv(data_file)
    print(f"✅ Loaded data for {df_raw['year'].nunique()} seasons")
else:
    print("🌐 Collecting F1 data from Ergast API...")
    print("⏳ This may take a few minutes...")
    
    # Collect data from 2010 to 2024
    df_raw = collector.get_seasons_data(2010, 2024)
    
    if not df_raw.empty:
        # Create data directory if it doesn't exist
        os.makedirs('../data', exist_ok=True)
        
        # Save the data
        df_raw.to_csv(data_file, index=False)
        print(f"💾 Data saved to {data_file}")
    else:
        print("❌ Failed to collect data")

# Display basic information about the dataset
print(f"\n📊 Dataset Overview:")
print(f"Shape: {df_raw.shape}")
print(f"Seasons: {df_raw['year'].min()} - {df_raw['year'].max()}")
print(f"Total driver records: {len(df_raw)}")
print(f"Unique drivers: {df_raw['driver_name'].nunique()}")
print(f"Unique constructors: {df_raw['constructor_name'].nunique()}")

df_raw.head()

## 3. Data Preprocessing and Feature Engineering

Transform raw F1 data into meaningful features for machine learning. This includes creating performance metrics, historical trends, and relative performance indicators.

In [ ]:
# Initialize feature engineer
engineer = F1FeatureEngineer()
print("⚙️ Feature Engineer initialized")

# Check for missing values in raw data
print("\n📋 Missing Values in Raw Data:")
missing_values = df_raw.isnull().sum()
print(missing_values[missing_values > 0])

# Basic data quality checks
print(f"\n🔍 Data Quality Checks:")
print(f"Duplicate records: {df_raw.duplicated().sum()}")
print(f"Records with zero points: {(df_raw['final_points'] == 0).sum()}")
print(f"Records with DNF rate > 50%: {(df_raw['dnf_rate'] > 0.5).sum() if 'dnf_rate' in df_raw.columns else 'N/A'}")

# Create features using our feature engineering module
print("\n🔧 Creating features...")
df_features = engineer.create_features(df_raw)

print(f"✅ Feature engineering complete!")
print(f"Original columns: {len(df_raw.columns)}")
print(f"After feature engineering: {len(df_features.columns)}")
print(f"New features created: {len(df_features.columns) - len(df_raw.columns)}")

# Display information about the engineered features
print(f"\n📊 Feature Summary:")
feature_categories = {
    'Performance': ['points_per_race', 'win_rate', 'podium_efficiency', 'race_craft_score'],
    'Historical': ['prev_year_points', 'years_in_f1', 'rolling_avg_points', 'career_total_wins'],
    'Relative': ['points_rank', 'points_percentile', 'points_gap_to_leader'],
    'Team': ['driver_team_points_share', 'team_competitiveness', 'constructor_final_position'],
    'Consistency': ['reliability_score', 'performance_consistency', 'championship_factor']
}

for category, features in feature_categories.items():
    available_features = [f for f in features if f in df_features.columns]
    print(f"{category}: {len(available_features)} features available")

df_features.head()

## 4. Exploratory Data Analysis

Analyze historical patterns and visualize key trends that influence championship outcomes.

In [ ]:
# Championship winners by year
champions = df_features[df_features['is_champion'] == 1].sort_values('year')

print("🏆 F1 World Champions (2010-2024):")
for _, champion in champions.iterrows():
    print(f"{champion['year']}: {champion['driver_name']} ({champion['constructor_name']}) - {champion['final_points']} pts")

# Visualize championship distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Points distribution for champions vs non-champions
axes[0,0].hist(df_features[df_features['is_champion'] == 0]['final_points'], 
               bins=30, alpha=0.7, label='Non-Champions', density=True)
axes[0,0].hist(df_features[df_features['is_champion'] == 1]['final_points'], 
               bins=15, alpha=0.7, label='Champions', density=True)
axes[0,0].set_xlabel('Final Points')
axes[0,0].set_ylabel('Density')
axes[0,0].set_title('Points Distribution: Champions vs Non-Champions')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 2. Win rate comparison
win_rate_champions = df_features[df_features['is_champion'] == 1]['win_rate']
win_rate_others = df_features[df_features['is_champion'] == 0]['win_rate']

axes[0,1].boxplot([win_rate_others, win_rate_champions], 
                  labels=['Non-Champions', 'Champions'])
axes[0,1].set_ylabel('Win Rate')
axes[0,1].set_title('Win Rate Distribution')
axes[0,1].grid(True, alpha=0.3)

# 3. Constructor dominance over time
constructor_champions = champions.groupby('constructor_name').size().sort_values(ascending=False)
axes[1,0].bar(constructor_champions.index, constructor_champions.values)
axes[1,0].set_xlabel('Constructor')
axes[1,0].set_ylabel('Championships Won')
axes[1,0].set_title('Constructor Championships (2010-2024)')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Championship points trend over years
yearly_champion_points = champions.groupby('year')['final_points'].first()
axes[1,1].plot(yearly_champion_points.index, yearly_champion_points.values, marker='o')
axes[1,1].set_xlabel('Year')
axes[1,1].set_ylabel('Champion Points')
axes[1,1].set_title('Championship Points Trend')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Key Statistics:")
print(f"Average champion points: {champions['final_points'].mean():.1f}")
print(f"Average champion wins: {champions['wins'].mean():.1f}")
print(f"Average champion win rate: {champions['win_rate'].mean():.1%}")
print(f"Most successful constructor: {constructor_champions.index[0]} ({constructor_champions.iloc[0]} titles)")

## 5. Feature Selection

Identify the most important features for predicting championship outcomes using statistical analysis and correlation.

In [ ]:
# Prepare data for training
target_col = 'is_champion'

# Get feature columns
feature_cols = engineer.select_features(df_features)
print(f"🎯 Selected {len(feature_cols)} features for modeling")

# Remove rows with missing target
clean_df = df_features.dropna(subset=[target_col])
print(f"📊 Clean dataset shape: {clean_df.shape}")

# Handle missing values in features
X = clean_df[feature_cols].fillna(clean_df[feature_cols].median())
y = clean_df[target_col]

print(f"\n📈 Target distribution:")
print(y.value_counts())
print(f"Champion rate: {y.mean():.1%}")

# Correlation analysis with target variable
correlations = X.corrwith(y).abs().sort_values(ascending=False)

# Display top correlated features
print(f"\n🔍 Top 15 Features Correlated with Championship:")
for i, (feature, corr) in enumerate(correlations.head(15).items(), 1):
    print(f"{i:2d}. {feature:<25}: {corr:.4f}")

# Visualize feature correlations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top correlations bar plot
top_corr = correlations.head(15)
axes[0].barh(range(len(top_corr)), top_corr.values)
axes[0].set_yticks(range(len(top_corr)))
axes[0].set_yticklabels(top_corr.index)
axes[0].set_xlabel('Absolute Correlation with Championship')
axes[0].set_title('Top 15 Feature Correlations')
axes[0].invert_yaxis()

# Correlation heatmap for top features
top_features = top_corr.head(10).index.tolist() + [target_col]
corr_matrix = clean_df[top_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Correlation Matrix - Top Features')

plt.tight_layout()
plt.show()

# Feature importance using Random Forest
print("\n🌳 Random Forest Feature Importance:")
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Features by Random Forest Importance:")
for i, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<25}: {row['importance']:.4f}")

# Store selected features for model training
selected_features = feature_cols
print(f"\n✅ Using {len(selected_features)} features for final model")

## 6. Model Training and Selection

Train multiple machine learning models and create an ensemble for robust predictions.

In [ ]:
# Initialize the ML predictor
predictor = F1ChampionshipPredictor(random_state=42)
print("🤖 F1 Championship Predictor initialized")

# Train the full model using our feature-engineered data
print("\n⏳ Training championship prediction model...")
print("This includes feature engineering, model training, and ensemble creation...")

training_results = predictor.train(df_features, target_col='is_champion', optimize_hyperparameters=True)

print("\n✅ Model training completed!")

# Display training results
print(f"\n📊 Model Performance Summary:")
print("="*50)

print(f"\nCross-Validation F1 Scores:")
for model_name, score in training_results['model_scores'].items():
    print(f"  {model_name:<20}: {score:.4f}")

print(f"\nTest Set Performance:")
for model_name, metrics in training_results['test_results'].items():
    print(f"\n{model_name}:")
    for metric, value in metrics.items():
        if value is not None:
            if metric in ['accuracy', 'precision', 'recall', 'f1']:
                print(f"  {metric.capitalize():<12}: {value:.1%}")
            else:
                print(f"  {metric.upper():<12}: {value:.4f}")

# Best performing model
best_model = max(training_results['model_scores'].items(), key=lambda x: x[1])
print(f"\n🏆 Best Individual Model: {best_model[0]} (F1: {best_model[1]:.4f})")

# Ensemble performance
if 'ensemble' in training_results['test_results']:
    ensemble_f1 = training_results['test_results']['ensemble']['f1']
    print(f"🎯 Ensemble Model F1 Score: {ensemble_f1:.4f}")

print(f"\n📈 Training Data Shape: {training_results['training_data_shape']}")
print(f"📉 Test Data Shape: {training_results['test_data_shape']}")

## 7. Model Evaluation

Evaluate model performance using comprehensive metrics and visualizations.

In [ ]:
# Initialize model evaluator
evaluator = F1ModelEvaluator()
print("📊 Model Evaluator initialized")

# Create comparison dataframe
comparison_df = evaluator.compare_models(training_results['test_results'])
print("\n🏆 Model Comparison Results:")
print(comparison_df.to_string(index=False))

# Visualize model comparison
print("\n📈 Generating performance comparison plots...")
evaluator.plot_model_comparison(comparison_df)
plt.show()

# Feature importance visualization
if not training_results['feature_importance'].empty:
    print("\n🔍 Top Feature Importance:")
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top_15_features = training_results['feature_importance'].head(15)
    
    bars = ax.barh(range(len(top_15_features)), top_15_features['importance'])
    ax.set_yticks(range(len(top_15_features)))
    ax.set_yticklabels(top_15_features['feature'])
    ax.set_xlabel('Feature Importance')
    ax.set_title('Top 15 Most Important Features for Championship Prediction')
    ax.invert_yaxis()
    
    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + 0.001, bar.get_y() + bar.get_height()/2.,
               f'{width:.3f}', ha='left', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Display top features
    print("\\n🎯 Top 10 Most Important Features:")
    for i, (_, row) in enumerate(top_15_features.head(10).iterrows(), 1):
        print(f"{i:2d}. {row['feature']:<25}: {row['importance']:.4f}")

# Generate comprehensive evaluation report
report = evaluator.generate_evaluation_report(
    training_results['test_results'], 
    save_path='../results/model_evaluation_report.txt'
)

print("\\n📋 Model Evaluation Report:")
print("="*60)
print(report)

# Save the trained model
model_path = '../models/f1_championship_predictor_trained.joblib'
os.makedirs('../models', exist_ok=True)
predictor.save_model(model_path)
print(f"\\n💾 Trained model saved to: {model_path}")

## 8. Prediction for 2025 Season

Apply the trained model to predict the 2025 F1 World Championship winner and visualize the results.

In [ ]:
# Load 2025 season data using our predictor interface
from f1_2025_predictor import F12025Predictor

# Initialize 2025 predictor
f1_2025 = F12025Predictor()
print("🏎️ F1 2025 Predictor initialized")

# Get 2025 season driver data
season_2025_data = f1_2025.f1_2025_drivers
print(f"\\n👥 2025 F1 Grid: {len(season_2025_data)} drivers confirmed")

# Use our trained model to make predictions
print("\\n🔮 Making 2025 championship predictions...")

# Transfer the trained model to the 2025 predictor
f1_2025.predictor = predictor

# Make predictions
predictions_2025 = predictor.predict_champion_probabilities(season_2025_data)

# Add betting odds and confidence levels
predictions_2025['odds'] = 1 / predictions_2025['championship_probability']
predictions_2025['confidence_level'] = pd.cut(
    predictions_2025['championship_probability'],
    bins=[0, 0.05, 0.15, 0.35, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

# Display results
print("\\n🏆 F1 2025 CHAMPIONSHIP PREDICTIONS")
print("="*60)

top_10 = predictions_2025.head(10)
for i, (_, row) in enumerate(top_10.iterrows(), 1):
    prob_pct = row['championship_probability'] * 100
    print(f"{i:2d}. {row['driver_name']:<20} ({row['constructor_name']:<12}) - "
          f"{prob_pct:6.2f}% (Odds: {row['odds']:5.1f}/1)")

# Visualize predictions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Top 10 championship probabilities
top_10_viz = predictions_2025.head(10)
bars1 = axes[0,0].bar(range(len(top_10_viz)), top_10_viz['championship_probability'] * 100)
axes[0,0].set_xticks(range(len(top_10_viz)))
axes[0,0].set_xticklabels(top_10_viz['driver_name'], rotation=45, ha='right')
axes[0,0].set_ylabel('Championship Probability (%)')
axes[0,0].set_title('Top 10 F1 2025 Championship Contenders')
axes[0,0].grid(True, alpha=0.3)

# Add percentage labels on bars
for bar, prob in zip(bars1, top_10_viz['championship_probability'] * 100):
    axes[0,0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                   f'{prob:.1f}%', ha='center', va='bottom', fontsize=8)

# 2. Constructor championship outlook
constructor_probs = predictions_2025.groupby('constructor_name')['championship_probability'].sum().sort_values(ascending=False)
axes[0,1].bar(constructor_probs.index, constructor_probs.values * 100)
axes[0,1].set_xlabel('Constructor')
axes[0,1].set_ylabel('Total Championship Probability (%)')
axes[0,1].set_title('Constructor Championship Outlook 2025')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)

# 3. Probability distribution
axes[1,0].hist(predictions_2025['championship_probability'] * 100, bins=15, alpha=0.7, edgecolor='black')
axes[1,0].set_xlabel('Championship Probability (%)')
axes[1,0].set_ylabel('Number of Drivers')
axes[1,0].set_title('Distribution of Championship Probabilities')
axes[1,0].grid(True, alpha=0.3)

# 4. Confidence levels
confidence_counts = predictions_2025['confidence_level'].value_counts()
wedges, texts, autotexts = axes[1,1].pie(confidence_counts.values, labels=confidence_counts.index, 
                                         autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Prediction Confidence Distribution')

plt.tight_layout()
plt.show()

# Championship favorite analysis
favorite = predictions_2025.iloc[0]
print(f"\\n🥇 Championship Favorite Analysis:")
print(f"Predicted Champion: {favorite['driver_name']}")
print(f"Team: {favorite['constructor_name']}")
print(f"Championship Probability: {favorite['championship_probability']:.1%}")
print(f"Betting Odds Equivalent: {favorite['odds']:.1f}/1")
print(f"Confidence Level: {favorite['confidence_level']}")

# Constructor analysis
print(f"\\n🏗️ Constructor Championship Outlook:")
for constructor, total_prob in constructor_probs.head(5).items():
    print(f"{constructor:<15}: {total_prob:.1%}")

# Save predictions
os.makedirs('../results', exist_ok=True)
predictions_path = '../results/f1_2025_predictions_notebook.csv'
predictions_2025.to_csv(predictions_path, index=False)
print(f"\\n💾 Predictions saved to: {predictions_path}")

print("\\n✅ F1 2025 Championship prediction completed!")
print("🏁 May the best driver win!")